# Definitions

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import sys, platform, os
os.environ['OMP_NUM_THREADS'] = '8'
import matplotlib
import math
from matplotlib import pyplot as plt
import numpy as np
import euclidemu2
import scipy
import cosmolike_lsst_y1_interface as ci
from getdist import IniFile
from scipy.interpolate import interp1d
import itertools
import iminuit
import functools
print(sys.version)
print(os.getcwd())

# GENERAL PLOT OPTIONS
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.family'] = 'STIXGeneral'
matplotlib.rcParams['mathtext.rm'] = 'Bitstream Vera Sans'
matplotlib.rcParams['mathtext.it'] = 'Bitstream Vera Sans:italic'
matplotlib.rcParams['mathtext.bf'] = 'Bitstream Vera Sans:bold'
matplotlib.rcParams['xtick.bottom'] = True
matplotlib.rcParams['xtick.top'] = False
matplotlib.rcParams['ytick.right'] = False
matplotlib.rcParams['axes.edgecolor'] = 'black'
matplotlib.rcParams['axes.linewidth'] = '1.0'
matplotlib.rcParams['axes.labelsize'] = 'medium'
matplotlib.rcParams['axes.grid'] = True
matplotlib.rcParams['grid.linewidth'] = '0.0'
matplotlib.rcParams['grid.alpha'] = '0.18'
matplotlib.rcParams['grid.color'] = 'lightgray'
matplotlib.rcParams['legend.labelspacing'] = 0.77
matplotlib.rcParams['savefig.bbox'] = 'tight'
matplotlib.rcParams['savefig.format'] = 'pdf'
# ----------------------------------------------------------------------------------------------------------------------------------
# ----------------------------------------------------------------------------------------------------------------------------------
# ----------------------------------------------------------------------------------------------------------------------------------
# BE CAREFUL: you need texlive-latex-base texlive-latex-extra texlive-fonts-recommended dvipng ghostscript cm-super
matplotlib.rcParams['text.usetex'] = True
# ----------------------------------------------------------------------------------------------------------------------------------
# ----------------------------------------------------------------------------------------------------------------------------------
# ----------------------------------------------------------------------------------------------------------------------------------

# Jupyter Notebook Display options
import IPython
IPython.display.display(IPython.display.HTML("<style>:root { --jp-notebook-max-width: 85% !important; }</style>"))
IPython.display.display(IPython.display.HTML("<style>div.output_scroll { height: 54em; }</style>"))

In [ ]:
# IMPORT CAMB
sys.path.insert(0, os.environ['ROOTDIR']+'/external_modules/code/CAMB/build/lib.linux-x86_64-'+os.environ['PYTHON_VERSION'])
import camb
from camb import model
print('Using CAMB %s installed at %s'%(camb.__version__,os.path.dirname(camb.__file__)))

# IMPORT SHARED NOTEBOOK UTILITIES (cosmolike_core)
sys.path.insert(0, os.environ['ROOTDIR']+'/external_modules/code/cosmolike_core')
import cosmolike_notebook_utils as cnu
print('Using cosmolike_notebook_utils installed at %s'%(os.path.dirname(cnu.__file__)))


# IMPORT THE PROJECT'S NOTEBOOK WRAPPERS (projects/lsst_y1/interface):
# the probe, chi2, Fisher, and bfmt wrappers every notebook of this
# project shares
import cosmolike_lsst_y1_notebook_wrappers as nw
print('Using notebook wrappers installed at %s'%(nw.__file__))

In [ ]:
CAMBAccuracyBoost = 1.0
non_linear_emul = 2
CLprobe="3x2pt"

path= "../../external_modules/data/lsst_y1"
data_file="lsst_y1_M1_GGL0.05.dataset"

IA_model = 0
IA_redshift_evolution = 3
IA_code = 0  # 0 = C FASTPT, 1 = Python FAST-PT (NLA always uses 0)

ntheta = 26
theta_min_arcmin = 2.5
theta_max_arcmin = 900
lmax = 50000


# hand this notebook's yaml-mirroring values to the shared wrappers
nw.configure(ntheta=ntheta,
             theta_min_arcmin=theta_min_arcmin,
             theta_max_arcmin=theta_max_arcmin,
             non_linear_emul=non_linear_emul)

In [ ]:
# The project fiducial point lives in the wrappers module
# (interface/cosmolike_lsst_y1_notebook_wrappers.py), shared by
# every notebook. The names are imported so the sweep cells below can
# build shifted vectors from them; override any value per call
# instead, e.g. nw.get_chi2(omegam=0.31).
from cosmolike_lsst_y1_notebook_wrappers import (
    As_1e9, ns, H0, omegab, omegam, mnu, w, w0pwa,
    LSST_A1_1, LSST_A1_2,
    LSST_DZ_S1, LSST_DZ_S2, LSST_DZ_S3, LSST_DZ_S4, LSST_DZ_S5,
    LSST_M1, LSST_M2, LSST_M3, LSST_M4, LSST_M5,
    LSST_DZ_L1, LSST_DZ_L2, LSST_DZ_L3, LSST_DZ_L4, LSST_DZ_L5,
    LSST_B1_1, LSST_B1_2, LSST_B1_3, LSST_B1_4, LSST_B1_5,
    LSST_PM_1, LSST_PM_2, LSST_PM_3, LSST_PM_4, LSST_PM_5)

In [ ]:
# Init Cosmolike for the masked 3x2pt data vector (the full
# sequence lives in nw.init_cosmolike)
ini = nw.init_cosmolike(CLprobe=CLprobe, with_data=True)

# The baryonic feedback methods of the `bfmt` theory block

The `bfmt` theory block (`external_modules/code/baryon_suppression`) implements seven ways to compute the suppression $S(k, z) = P_{\rm hydro}/P_{\rm DMO}$ of the nonlinear matter power spectrum: three semi-analytical SP(k) baryon-fraction relations and four emulators. Instead of re-implementing any of them here, this notebook drives the block itself through a minimal Cobaya model (CAMB + `bfmt` + the unit likelihood), requesting the suppression on this notebook's own $(k, z)$ interpolation grid - the same product, produced by the same code, that the Cosmolike likelihoods consume in an MCMC run.

In [ ]:
# One entry per feedback method: (label, bfmt options, fixed parameter
# point). The SP(k) points are pyspk's documented examples; the emulator
# points are the fiducial values quoted in the example yamls. BACCOemu
# is commented out on this project: its omega_baryon training box
# starts at 0.04001, exactly above this notebook's omegab = 0.04, so
# the block (correctly) rejects the fiducial.
BARYON_METHODS = [
    ("SP(k) power law", {"baryon_model": 1, "spk_fb_model": 1},
     {"fb_a_spk": 0.4, "fb_pow_spk": 0.3}),
    ("SP(k) Akino et al. 2022", {"baryon_model": 1, "spk_fb_model": 2},
     {"alpha_spk": 4.189, "beta_spk": 1.273, "gamma_spk": 0.298}),
    # this point matches the Akino relation's amplitude at the pivot
    # and stays inside SP(k)'s calibrated baryon-fraction band over
    # the full z grid; pyspk's documented example (0.3, 1.1, 0.2,
    # 0.5) exits the band at z >~ 1.4, and the block then falls back
    # to unity for those redshifts
    ("SP(k) double power law", {"baryon_model": 1, "spk_fb_model": 3},
     {"epsilon_spk": 0.66, "alpha_spk": 0.35, "beta_spk": 0.2,
      "gamma_spk": 0.3}),
    ("BCEmu", {"baryon_model": 2},
     {"log10Mc_bcemu": 13.32, "mu_bcemu": 0.93, "thej_bcemu": 4.235,
      "gamma_bcemu": 2.25, "delta_bcemu": 6.40, "eta_bcemu": 0.15,
      "deta_bcemu": 0.14}),
    ("Flamingo", {"baryon_model": 3},
     {"fgas_sigma_flamingo": 0.0, "mstar_sigma_flamingo": 0.0,
      "jet_frac_flamingo": 0.0}),
    #("BACCOemu", {"baryon_model": 4},
    # {"M_c_baccoemu": 14.0, "eta_baccoemu": -0.3, "beta_baccoemu": -0.22,
    #  "M1_z0_cen_baccoemu": 10.5, "theta_inn_baccoemu": -0.86}),
    ("BCemu2025", {"baryon_model": 5},
     {"Theta_co_bcemu25": 0.3, "log10Mc_bcemu25": 13.1, "mu_bcemu25": 1.0,
      "delta_bcemu25": 6.0, "eta_bcemu25": 0.10, "deta_bcemu25": 0.22,
      "Nstar_bcemu25": 0.028}),
]


In [ ]:
# get_baryon_suppression lives in this project's interface/
# (cosmolike_lsst_y1_notebook_wrappers, imported above as nw),
# shared by every notebook of this project; call as
# nw.get_baryon_suppression.

In [ ]:
# compute_probes lives in this project's interface/
# (cosmolike_lsst_y1_notebook_wrappers, imported above as nw),
# shared by every notebook of this project; call as
# nw.compute_probes.

# Compute the data vector without and with each feedback method

In [ ]:
ref = nw.compute_probes()
print("no feedback: chi2 = %.6f, data vector length = %d" % (ref["chi2"], len(ref["dv"])))

In [ ]:
results = {}
for label, theory_options, point in BARYON_METHODS:
    sup = nw.get_baryon_suppression(theory_options, point,
                                 ref["z_grid"], ref["log10k_grid"])
    results[label] = nw.compute_probes(sup = sup)
    print("%-28s chi2 = %10.4f" % (label, results[label]["chi2"]),
          flush=True)


# Print the data vectors

One row per masked entry: the dark-matter-only prediction first, then one column per feedback method. The chi2 line above each column shows how far each suppression pulls the prediction from the stored (feedback-free) data vector.

In [ ]:
chains_dir = os.environ["ROOTDIR"] + "/projects/lsst_y1/chains"
os.makedirs(chains_dir, exist_ok=True)
table = np.column_stack([ref["dv"]] +
                        [results[label]["dv"] for label, _, _ in BARYON_METHODS])
header = "no_feedback " + " ".join(
    label.replace(" ", "_") for label, _, _ in BARYON_METHODS)
np.savetxt(chains_dir + "/EXAMPLE_EVALUATE3.datavectors",
           table, header=header)
print(header)
for i in (0, 1, 2, len(table)//2, len(table)-2, len(table)-1):
    print(("%6d" % i) + "".join(" %.6e" % v for v in table[i]))
print("full table saved to " + chains_dir + "/EXAMPLE_EVALUATE3.datavectors")


# Cosmic shear with feedback: $C_\ell^{\rm EE}$ (Fourier space)

In [ ]:
labels = [label for label, _, _ in BARYON_METHODS]

param = np.arange(1.0, 1.0 + len(labels))

C_ss_list = [results[label]["C_ss"] for label in labels]

cnu.plot_C_ss_tomo_limber(ell=ref["ell"],
                          C_ss=C_ss_list,
                          C_ss_ref=ref["C_ss"],
                          param=param,
                          lmin=25,
                          lmax=3000,
                          ylim=(0.82, 1.05),
                          colorbarlabel="bfmt method",
                          legend=labels,
                          figsize=(18, 18),
                          legendloc=(0.55, 0.68),
                          legendfontsize=20,
                          xaxisticklabelsize=20,
                          yaxisticklabelsize=15,
                          yaxislabelsize=15)

# Cosmic shear with feedback: $\xi_\pm(\theta)$ (real space)

In [ ]:
xi_list = [(results[label]["theta"], results[label]["xip"],
            results[label]["xim"]) for label in labels]

xi_ref = (ref["theta"], ref["xip"], ref["xim"])

cnu.plot_xi(pm=1,
            xi=xi_list,
            xi_ref=xi_ref,
            param=param,
            ylim=(0.875, 1.05),
            colorbarlabel="bfmt method",
            legend=labels,
            thetashow=[2.5, 900],
            legendloc=(0.55, 0.68),
            legendfontsize=20,
            xaxisticklabelsize=20,
            yaxisticklabelsize=15,
            yaxislabelsize=15)

In [ ]:
cnu.plot_xi(pm=0,
            xi=xi_list,
            xi_ref=xi_ref,
            param=param,
            ylim=(0.875, 1.05),
            colorbarlabel="bfmt method",
            legend=labels,
            figsize=(18, 18),
            thetashow=[2.5, 900],
            legendloc=(0.55, 0.68),
            legendfontsize=20,
            xaxisticklabelsize=20,
            yaxisticklabelsize=15,
            yaxislabelsize=15)

# Galaxy-galaxy lensing with feedback: $C_\ell^{gs}$ (Fourier space)

In [ ]:
C_gs_list = [results[label]["C_gs"] for label in labels]

cnu.plot_C_gs_tomo_limber(ell=ref["ell"],
                          C_gs=C_gs_list,
                          C_gs_ref=ref["C_gs"],
                          param=param,
                          lmin=25,
                          lmax=3000,
                          figsize=(18, 18),
                          ylim=(0.8, 1.05),
                          colorbarlabel="bfmt method",
                          legend=labels,
                          legendfontsize=20,
                          xaxisticklabelsize=20,
                          yaxisticklabelsize=15,
                          yaxislabelsize=15)

# Galaxy-galaxy lensing with feedback: $\gamma_t(\theta)$ (real space)

In [ ]:
gammat_list = [(results[label]["theta"], results[label]["gammat"])
               for label in labels]
gammat_ref = (ref["theta"], ref["gammat"])
cnu.plot_gammat_tomo_limber(theta_gammat=gammat_list,
                            gammat_ref=gammat_ref,
                            param=param,
                            ylim=(0.8, 1.05),
                            figsize=(18, 18),
                            colorbarlabel="bfmt method",
                            legend=labels,
                            thetashow=[2.5, 900],
                            legendfontsize=20,
                            xaxisticklabelsize=20,
                            yaxisticklabelsize=15,
                            yaxislabelsize=15)

# Vary each method's free parameters, one at a time

Each figure below sweeps a single parameter of one feedback method,
holding everything else at the method's fiducial point, and shows the
suppression $S(k) = P_{\rm feedback}/P_{\rm DM}$ at $z = 0$ (solid) and
$z = 1$ (dashed). The sweep ranges sit inside the bfmt validation boxes
(the 3$\sigma$ boxes of the yaml priors for SP(k) Akino, the emulator
training boxes otherwise). An SP(k) curve can revert to unity at a
redshift where the swept parameter pushes the baryon fraction outside
pyspk's calibrated band — that is the block's documented fallback, not a
bug. Every value is one `get_baryon_suppression` call (a fresh CAMB run),
so this section evaluates CAMB ~130 times and takes tens of minutes.

In [ ]:
import io
from contextlib import redirect_stdout, redirect_stderr

# The sweep grid: two redshifts are enough to see the evolution, and
# the k grid spans the scales where feedback acts (1/Mpc, the unit the
# likelihoods send). SWEEP_RESULTS keeps every computed sweep, so a
# figure can be replotted without recomputing.
z_sweep = [0.0, 1.0]
log10k_sweep = np.linspace(-1.3, 1.1, 120)
SWEEP_RESULTS = {}
METHODS_BY_LABEL = {label: (options, point)
                    for label, options, point in BARYON_METHODS}

PARAM_SWEEPS = {
    # loose boxes; the hard guard is pyspk's calibrated fb band
    "SP(k) power law": {
        "fb_a_spk":   [0.2, 0.3, 0.4, 0.5, 0.6],
        "fb_pow_spk": [0.1, 0.2, 0.3, 0.4, 0.5],
    },
    # 3-sigma boxes: alpha 3.8-4.6, beta 1.0-1.6, gamma 0.1-0.75
    "SP(k) Akino et al. 2022": {
        "alpha_spk": [3.9, 4.05, 4.189, 4.35, 4.5],
        "beta_spk":  [1.05, 1.16, 1.273, 1.4, 1.55],
        "gamma_spk": [0.15, 0.22, 0.298, 0.45, 0.6],
    },
    # centered on the in-band point of the methods table above
    "SP(k) double power law": {
        "epsilon_spk": [0.50, 0.58, 0.66, 0.74, 0.82],
        "alpha_spk":   [0.15, 0.25, 0.35, 0.45, 0.55],
        "beta_spk":    [0.10, 0.15, 0.20, 0.25, 0.30],
        "gamma_spk":   [0.10, 0.20, 0.30, 0.40, 0.50],
    },
    # inside the Giri et al. 2021 training box
    "BCEmu": {
        "log10Mc_bcemu": [12.5, 13.0, 13.32, 13.8, 14.3],
        "mu_bcemu":      [0.3, 0.6, 0.93, 1.3, 1.7],
        "thej_bcemu":    [2.5, 3.3, 4.235, 5.3, 6.5],
        "gamma_bcemu":   [1.5, 1.9, 2.25, 2.8, 3.5],
        "delta_bcemu":   [4.0, 5.2, 6.4, 7.7, 9.0],
        "eta_bcemu":     [0.08, 0.11, 0.15, 0.25, 0.35],
        "deta_bcemu":    [0.08, 0.11, 0.14, 0.25, 0.35],
    },
    # the Schaller et al. 2025 suite spans fgas -8sigma..+2sigma,
    # mstar -1sigma, and jet fractions 0..1
    "Flamingo": {
        "fgas_sigma_flamingo":  [-8.0, -4.0, -2.0, 0.0, 2.0],
        "mstar_sigma_flamingo": [-1.0, -0.5, 0.0, 0.5, 1.0],
        "jet_frac_flamingo":    [0.0, 0.25, 0.5, 0.75, 1.0],
    },
    # inside the BCemu2025 training LHC box
    "BCemu2025": {
        "Theta_co_bcemu25": [0.1, 0.2, 0.3, 0.45, 0.6],
        "log10Mc_bcemu25":  [12.0, 12.6, 13.1, 13.7, 14.4],
        "mu_bcemu25":       [0.4, 0.7, 1.0, 1.5, 2.0],
        "delta_bcemu25":    [3.0, 4.5, 6.0, 8.0, 10.0],
        "eta_bcemu25":      [-0.15, -0.05, 0.02, 0.10, 0.18],
        "deta_bcemu25":     [0.05, 0.12, 0.22, 0.30, 0.38],
        "Nstar_bcemu25":    [0.010, 0.019, 0.028, 0.037, 0.045],
    },
}

def sweep_parameter(label, name, values):
    """One-at-a-time sweep: name takes each value, the rest stay at
    the method's fiducial point; returns one (n_z, n_k) array per
    value and caches them in SWEEP_RESULTS. Everything the libraries
    print while a model builds (emulator loading bars, cobaya
    banners) goes to an in-memory buffer instead of the notebook, so
    a sweep cell shows only its figures; an exception still surfaces
    with its full traceback."""
    options, fiducial = METHODS_BY_LABEL[label]
    sups = []
    for v in values:
        point = dict(fiducial, **{name: v})
        chatter = io.StringIO()
        with redirect_stdout(chatter), redirect_stderr(chatter):
            S = nw.get_baryon_suppression(options, point,
                                       z_sweep, log10k_sweep)
        sups.append(np.array([S[z] for z in z_sweep]))
    SWEEP_RESULTS.setdefault(label, {})[name] = sups
    return sups

def plot_sweeps(label):
    for name, values in PARAM_SWEEPS[label].items():
        sups = sweep_parameter(label, name, values)
        cnu.plot_baryon_suppression(log10k=log10k_sweep,
                                    sup=sups,
                                    param=np.array(values),
                                    colorbarlabel=name.replace("_", " "),
                                    zlabels=["$z=0$", "$z=1$"],
                                    ylim=(0.6, 1.35),
                                    title=label + ": vary " + name.replace("_", " "))

In [ ]:
plot_sweeps("SP(k) power law")

In [ ]:
plot_sweeps("SP(k) Akino et al. 2022")

In [ ]:
plot_sweeps("SP(k) double power law")

In [ ]:
plot_sweeps("BCEmu")

In [ ]:
plot_sweeps("Flamingo")

In [ ]:
plot_sweeps("BCemu2025")